# Exploring the Confabulation Probe

This notebook lets you interactively explore the probe, the activations, and the trajectories.
It runs **locally** (no GPU needed) since all activations are pre-extracted `.npy` files.

You can:
- Visualize the layer sweep
- Inspect individual trajectories and their probe scores
- Retrain the probe with different hyperparameters
- Explore what makes the model fabricate vs admit
- Test the probe on new splits or subsets

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path('..') if Path('../config.yaml').exists() else Path('.')
random.seed(42)
np.random.seed(42)

# Load everything
with open(ROOT / 'data/trajectories/all_trajectories.json') as f:
    all_trajs = json.load(f)
with open(ROOT / 'data/labels.json') as f:
    label_data = json.load(f)

judge = label_data['judge_labels']
cal_ids = set(label_data['calibration_ids'])

print(f'Loaded {len(all_trajs)} trajectories, {len(judge)} labels')

## 1. Data Overview

Look at the distribution of trajectories by side, prompt, and label.

In [ ]:
# Heuristic labeler (matches the regex used throughout the project)
def heuristic_label(resp):
    number_pat = re.compile(r'\b\d+\.?\d*\s*(?:eV|g/cm|meV|J|kJ|GPa)')
    estimate_pat = re.compile(r'(?:approximately|around|roughly|about|estimated?|~)\s*\d+\.?\d*', re.I)
    range_pat = re.compile(r'\d+\.?\d*\s*[-\u2013to]+\s*\d+\.?\d*\s*(?:eV|g/cm|meV)', re.I)
    return 1 if (number_pat.search(resp) or estimate_pat.search(resp) or range_pat.search(resp)) else 0

# Build a clean dataset
empty = [t for t in all_trajs if t['side'] == 'empty']
present = [t for t in all_trajs if t['side'] == 'data_present']

print(f'Empty-side: {len(empty)}')
print(f'Data-present: {len(present)}')
print()

print('Fabrication rate by system prompt (heuristic labels):')
for sp in ['neutral', 'pressure', 'honesty', 'balanced', 'expert']:
    sub = [t for t in empty if t['system_prompt_variant'] == sp]
    fab = sum(heuristic_label(t['assistant_response']) for t in sub)
    print(f'  {sp:12s}: {fab:3d}/{len(sub)} ({fab/len(sub):.1%})')

## 2. Layer Sweep Visualization

Train a probe at each of the 33 layers and plot test AUROC. This is the
core result of the project: a non-monotonic curve peaking at layer 16.

In [ ]:
def load_act(tid, layer):
    return np.load(ROOT / f'data/trajectories/activations/{tid}.npy')[layer]

# Balanced-only data with heuristic labels
bal = [t for t in empty if t['system_prompt_variant'] == 'balanced']
for t in bal:
    t['label'] = heuristic_label(t['assistant_response'])

# Material-based split
formulas = sorted(set(t['formula'] for t in bal))
random.shuffle(formulas)
n_train = int(0.7 * len(formulas))
train_f = set(formulas[:n_train])
test_f = set(formulas[n_train:])

train = [t for t in bal if t['formula'] in train_f]
test = [t for t in bal if t['formula'] in test_f]

print(f'Train: {len(train)} trajectories from {len(train_f)} materials')
print(f'Test:  {len(test)} trajectories from {len(test_f)} materials')

# Sweep
train_aurocs, test_aurocs = [], []
for layer in range(33):
    tr_X = np.stack([load_act(t['trajectory_id'], layer) for t in train])
    tr_y = np.array([t['label'] for t in train])
    te_X = np.stack([load_act(t['trajectory_id'], layer) for t in test])
    te_y = np.array([t['label'] for t in test])

    clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
    clf.fit(tr_X, tr_y)

    tr_auroc = roc_auc_score(tr_y, clf.predict_proba(tr_X)[:, 1])
    te_auroc = roc_auc_score(te_y, clf.predict_proba(te_X)[:, 1])
    train_aurocs.append(tr_auroc)
    test_aurocs.append(te_auroc)

best_layer = np.argmax(test_aurocs)
print(f'\nBest layer: {best_layer}, test AUROC: {test_aurocs[best_layer]:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
layers = list(range(33))
ax.plot(layers, train_aurocs, 'o-', label='Train', alpha=0.7)
ax.plot(layers, test_aurocs, 's-', label='Test', alpha=0.7)
ax.axvline(best_layer, color='red', linestyle='--', alpha=0.5, label=f'Best (layer {best_layer})')
ax.axhline(0.75, color='gray', linestyle=':', alpha=0.5, label='Success threshold (0.75)')
ax.set_xlabel('Layer')
ax.set_ylabel('AUROC')
ax.set_title('Layer Sweep: Fabrication Probe (Balanced Prompt, Material-Split)')
ax.legend()
ax.set_ylim(0.45, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. ROC Curve at Best Layer

The ROC curve shows the tradeoff between true positive rate (catching
fabrications) and false positive rate (incorrectly flagging admissions).

In [ ]:
te_X = np.stack([load_act(t['trajectory_id'], best_layer) for t in test])
te_y = np.array([t['label'] for t in test])
clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
tr_X = np.stack([load_act(t['trajectory_id'], best_layer) for t in train])
tr_y = np.array([t['label'] for t in train])
clf.fit(tr_X, tr_y)
proba = clf.predict_proba(te_X)[:, 1]

fpr, tpr, thresholds = roc_curve(te_y, proba)
auroc = roc_auc_score(te_y, proba)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'Probe (AUROC={auroc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve (Layer {best_layer})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. Inspect Individual Trajectories

Look at specific trajectories, their probe scores, and their responses.
This helps build intuition for what the model fabricates vs admits.

In [ ]:
# Score every test trajectory
scored = []
for i, t in enumerate(test):
    scored.append({
        'id': t['trajectory_id'],
        'formula': t['formula'],
        'property': t['property'],
        'template_index': t['template_index'],
        'label': 'FABRICATE' if t['label'] == 1 else 'ADMIT',
        'probe_score': float(proba[i]),
        'response': t['assistant_response'][:300],
    })

scored.sort(key=lambda x: x['probe_score'], reverse=True)

print('TOP 5 (highest probe score, most likely to fabricate):')
for s in scored[:5]:
    print(f"  [{s['probe_score']:.3f}] {s['label']:9s} | {s['formula']} | {s['property']}")
    print(f"    {s['response'][:150]}")
    print()

print('BOTTOM 5 (lowest probe score, most likely to admit):')
for s in scored[-5:]:
    print(f"  [{s['probe_score']:.3f}] {s['label']:9s} | {s['formula']} | {s['property']}")
    print(f"    {s['response'][:150]}")
    print()

## 5. Fabrication Patterns

Explore what makes fabrication more likely: which properties, templates,
and formula characteristics correlate with fabrication?

In [ ]:
print('Fabrication rate by property (balanced prompt):')
for prop in ['band_gap', 'formation_energy_per_atom', 'density']:
    sub = [t for t in bal if t['property'] == prop]
    fab = sum(t['label'] for t in sub)
    print(f'  {prop:30s}: {fab}/{len(sub)} ({fab/len(sub):.1%})')

print()
print('Fabrication rate by template index (balanced prompt):')
for ti in range(5):
    sub = [t for t in bal if t['template_index'] == ti]
    fab = sum(t['label'] for t in sub)
    # Show the template text for one example
    example = sub[0]['user_query'] if sub else ''
    print(f'  Template {ti}: {fab}/{len(sub)} ({fab/len(sub):.1%})')
    print(f'    Example: {example[:80]}')

print()
print('Fabrication rate by formula length (number of elements):')
def count_elements(formula):
    return len(set(el for el, _ in re.findall(r'([A-Z][a-z]?)(\d*)', formula) if el))

for n_el in range(2, 8):
    sub = [t for t in bal if count_elements(t['formula']) == n_el]
    if not sub:
        continue
    fab = sum(t['label'] for t in sub)
    print(f'  {n_el} elements: {fab}/{len(sub)} ({fab/len(sub):.1%})')

## 6. Retrain with Different Hyperparameters

Try changing the regularization strength C, or use a different
classifier entirely. The activations are already extracted, so
this is fast.

In [ ]:
# Sweep regularization strength C
print(f'Regularization sweep at layer {best_layer}:')
print(f'{"C":>8s}  {"Train":>8s}  {"Test":>8s}')
for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0]:
    clf_c = LogisticRegression(C=C, max_iter=2000, solver='lbfgs')
    clf_c.fit(tr_X, tr_y)
    tr_auc = roc_auc_score(tr_y, clf_c.predict_proba(tr_X)[:, 1])
    te_auc = roc_auc_score(te_y, clf_c.predict_proba(te_X)[:, 1])
    print(f'{C:8.2f}  {tr_auc:8.4f}  {te_auc:8.4f}')

## 7. Probe Direction Visualization

Project the test-set activations onto the top 2 principal components
plus the probe direction. This shows how the fabrication and admission
clusters are arranged in activation space.

In [ ]:
from sklearn.decomposition import PCA

all_X = np.vstack([tr_X, te_X])
all_y = np.concatenate([tr_y, te_y])

# Project onto probe direction + top PCA component orthogonal to it
direction = clf.coef_[0]
direction_norm = direction / np.linalg.norm(direction)

proj_probe = all_X @ direction_norm

# Residual after removing probe direction
residual = all_X - np.outer(proj_probe, direction_norm)
pca = PCA(n_components=1)
proj_pca = pca.fit_transform(residual).ravel()

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
colors = ['#2196F3' if y == 0 else '#F44336' for y in all_y]
ax.scatter(proj_probe, proj_pca, c=colors, alpha=0.5, s=30)
ax.set_xlabel('Projection onto probe direction')
ax.set_ylabel('Top PCA component (orthogonal to probe)')
ax.set_title('Activations at Layer 16: Admit (blue) vs Fabricate (red)')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2196F3', markersize=8, label='Admit'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#F44336', markersize=8, label='Fabricate'),
]
ax.legend(handles=legend_elements)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Intervention Simulation

Using the saved intervention results, visualize the fabrication rate
under different interventions.

In [ ]:
with open(ROOT / 'data/intervention_results.json') as f:
    intv = json.load(f)

alphas = sorted(intv['steering'].keys(), key=float)
steer_rates = [intv['steering'][a]['fab_rate'] for a in alphas]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# Baseline
ax.axhline(intv['baseline_fab_rate'], color='gray', linestyle='--', label='Baseline (no intervention)')

# Prompt injection
ax.axhline(intv['injection_fab_rate'], color='green', linestyle='-', linewidth=2, label='Prompt injection')

# Steering sweep
ax.plot([float(a) for a in alphas], steer_rates, 'ro-', label='Activation steering')

ax.set_xlabel('Steering strength (\u03b1)')
ax.set_ylabel('Fabrication rate')
ax.set_title('Intervention Effectiveness')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()

print(f'Baseline fabrication:  {intv["baseline_fab_rate"]:.1%}')
print(f'Prompt injection:     {intv["injection_fab_rate"]:.1%} (reduction: {intv["injection_reduction"]:.1%})')
print(f'Accuracy preserved:   {intv["injection_accuracy"]:.1%} (baseline: {intv["baseline_accuracy"]:.1%})')

## 9. Extending the Project

Ideas for further exploration:

- **Different layers for steering.** The probe peaks at layer 16, but
  the best layer for causal intervention might be different. Try
  steering at layers 10-20 and compare.

- **Nonlinear probes.** Replace logistic regression with a small MLP
  or random forest. Does nonlinearity help?

- **Attention pattern analysis.** Instead of probing the residual stream,
  look at attention weights. Do fabricating trajectories attend differently
  to the empty tool result?

- **Other models.** Extract activations from Llama-3.1-70B or a different
  model family and test whether the same layer-16 direction transfers.

- **Temperature > 0.** All trajectories use greedy decoding. Running with
  temperature sampling would produce a more diverse distribution of
  responses per material, potentially improving probe robustness.